[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/08-email-notifications.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/08-email-notifications.ipynb)

# Module 3.8 — Email & Notifications
**Module 3: Automation & Scripting** | Estimated time: 30 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Build and send plain-text and HTML emails using `smtplib` and the `email` module
- Attach files to emails with `MIMEBase` and `encoders`
- Authenticate with Gmail using an App Password
- Follow best practices for credential management (`os.environ`)
- Understand the structure for sending SMS via Twilio (requires real credentials)
- Write a reusable, production-ready email-sending function with full error handling

In [ ]:
!pip install twilio -q

import smtplib
import ssl
import os
import json
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email.mime.application import MIMEApplication
from email import encoders
from email.utils import formataddr, formatdate, make_msgid
from pathlib import Path
import datetime

print('Standard library email modules ready.')
print('twilio imported:', __import__('twilio').__version__)

## 1. Credential Best Practices

**Never hard-code passwords or API keys in source code.**  
In Google Colab, use the Secrets panel (the key icon in the left sidebar) and access them via `userdata.get()`. In a local script, use environment variables.

```
# In a local script
EMAIL_PASSWORD = os.environ.get('GMAIL_APP_PASSWORD')

# In Google Colab
from google.colab import userdata
EMAIL_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
```

For Gmail specifically you need an **App Password**, not your main password:
1. Enable 2-Factor Authentication on your Google account
2. Go to `myaccount.google.com` > Security > App passwords
3. Create an app password for "Mail" + "Windows Computer"
4. Use the 16-character app password in your script

In [ ]:
# Load credentials from environment (set these before running!)
# In Colab: Runtime > Manage Keys  (or skip sending and just inspect the message objects)
SMTP_HOST     = os.environ.get('SMTP_HOST',     'smtp.gmail.com')
SMTP_PORT     = int(os.environ.get('SMTP_PORT', '465'))  # 465=SSL, 587=STARTTLS
SMTP_USER     = os.environ.get('SMTP_USER',     'your_email@gmail.com')
SMTP_PASSWORD = os.environ.get('SMTP_PASSWORD', '')       # Gmail App Password
FROM_NAME     = os.environ.get('FROM_NAME',     'PyPath Bot')

CREDENTIALS_SET = bool(SMTP_PASSWORD and SMTP_USER != 'your_email@gmail.com')
print('Credentials configured:', CREDENTIALS_SET)
if not CREDENTIALS_SET:
    print('NOTE: Set SMTP_USER and SMTP_PASSWORD env vars to actually send emails.')
    print('The rest of this notebook demonstrates message construction.')

## 2. Building a Plain-Text Email

In [ ]:
def build_plain_email(to_addr: str, subject: str, body: str) -> MIMEText:
    """Create a simple plain-text email."""
    msg = MIMEText(body, 'plain', 'utf-8')
    msg['From']       = formataddr((FROM_NAME, SMTP_USER))
    msg['To']         = to_addr
    msg['Subject']    = subject
    msg['Date']       = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid(domain='pypath.dev')
    return msg


plain_msg = build_plain_email(
    to_addr='recipient@example.com',
    subject='Daily Scraping Report — 2024-03-15',
    body=(
        'Hi,\n\n'
        'Today\'s scraping job has completed successfully.\n\n'
        'Summary:\n'
        '  Pages scraped: 50\n'
        '  Items found  : 1000\n'
        '  Errors       : 0\n\n'
        'Best regards,\nPyPath Bot'
    ),
)

print('Plain-text email headers:')
for key in ['From', 'To', 'Subject', 'Date', 'Message-ID']:
    print(f'  {key}: {plain_msg[key]}')
print('\nBody preview:')
print(plain_msg.get_payload()[:200])

## 3. Building an HTML Email with Plain-Text Fallback

A proper HTML email always includes a plain-text alternative. Email clients that cannot render HTML will display the plain-text part.

In [ ]:
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <style>
    body {{ font-family: Arial, sans-serif; color: #333; max-width: 600px; }}
    .header {{ background: #2F75B6; color: white; padding: 20px; border-radius: 4px 4px 0 0; }}
    .body   {{ padding: 20px; border: 1px solid #ddd; border-top: none; }}
    .stat   {{ display: inline-block; margin: 10px; padding: 15px 25px;
              background: #f0f8ff; border-radius: 4px; text-align: center; }}
    .stat strong {{ display: block; font-size: 2em; color: #2F75B6; }}
    .footer {{ font-size: 11px; color: #888; padding: 10px 20px; }}
  </style>
</head>
<body>
  <div class="header"><h2>PyPath Daily Report</h2></div>
  <div class="body">
    <p>Hi {name},</p>
    <p>Your scraping job <strong>{job_name}</strong> completed on {date}.</p>
    <div class="stat"><strong>{pages}</strong>Pages Scraped</div>
    <div class="stat"><strong>{items}</strong>Items Found</div>
    <div class="stat"><strong>{errors}</strong>Errors</div>
    <p>View full results in the dashboard.</p>
  </div>
  <div class="footer">PyPath Automation &mdash; You are receiving this because you set up a scraping job.</div>
</body>
</html>
"""

def build_html_email(to_addr: str, subject: str,
                     plain_text: str, html_text: str) -> MIMEMultipart:
    """Create a multipart/alternative email with HTML and plain-text parts."""
    msg = MIMEMultipart('alternative')
    msg['From']       = formataddr((FROM_NAME, SMTP_USER))
    msg['To']         = to_addr
    msg['Subject']    = subject
    msg['Date']       = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid(domain='pypath.dev')
    # Plain text first (lower preference), HTML second (higher preference)
    msg.attach(MIMEText(plain_text, 'plain', 'utf-8'))
    msg.attach(MIMEText(html_text,  'html',  'utf-8'))
    return msg


html_body = HTML_TEMPLATE.format(
    name='Alice', job_name='books_catalogue',
    date='2024-03-15', pages=50, items=1000, errors=0
)
plain_body = 'Job: books_catalogue | Pages: 50 | Items: 1000 | Errors: 0'

html_msg = build_html_email(
    to_addr='alice@example.com',
    subject='PyPath Report: books_catalogue (2024-03-15)',
    plain_text=plain_body,
    html_text=html_body,
)
print('HTML email structure:')
for part in html_msg.walk():
    print(f'  Content-Type: {part.get_content_type()}')
print(f'\nHTML body length: {len(html_body)} chars')

## 4. Attaching Files to an Email

In [ ]:
def add_attachment(msg: MIMEMultipart, file_path: str, filename: str = None) -> None:
    """Attach a file to an existing MIMEMultipart message."""
    path = Path(file_path)
    filename = filename or path.name
    with open(path, 'rb') as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', 'attachment', filename=filename)
    msg.add_header('MIME-Version', '1.0')  # ensure header present
    # Convert to mixed for attachment
    msg.attach(part)


# Create a test attachment file
report_path = '/tmp/report.csv'
with open(report_path, 'w') as f:
    f.write('date,pages,items\n2024-03-15,50,1000\n2024-03-14,48,960\n')

# Build a mixed message (body + attachment)
mixed_msg = MIMEMultipart('mixed')
mixed_msg['From']    = formataddr((FROM_NAME, SMTP_USER))
mixed_msg['To']      = 'alice@example.com'
mixed_msg['Subject'] = 'Report with Attachment'
mixed_msg['Date']    = formatdate(localtime=True)

# Embed the alternative part as a sub-part
alternative = MIMEMultipart('alternative')
alternative.attach(MIMEText(plain_body, 'plain', 'utf-8'))
alternative.attach(MIMEText(html_body,  'html',  'utf-8'))
mixed_msg.attach(alternative)

add_attachment(mixed_msg, report_path)

print('Mixed message parts:')
for part in mixed_msg.walk():
    disp = part.get('Content-Disposition', '')
    print(f'  {part.get_content_type():<30s} {disp[:40]}')

## 5. Sending with `smtplib` — SSL and STARTTLS

In [ ]:
def send_email(msg, smtp_host=SMTP_HOST, smtp_port=SMTP_PORT,
               username=SMTP_USER, password=SMTP_PASSWORD) -> bool:
    """
    Send an email message.
    Uses SSL (port 465) or STARTTLS (port 587) automatically.
    Returns True on success, False on failure.
    """
    if not password:
        print('[SEND SKIPPED] No SMTP_PASSWORD set. Message was built successfully.')
        print(f'  To: {msg["To"]}  |  Subject: {msg["Subject"]}')
        return False

    try:
        context = ssl.create_default_context()
        if smtp_port == 465:
            # SSL from the start
            with smtplib.SMTP_SSL(smtp_host, smtp_port, context=context) as server:
                server.login(username, password)
                server.send_message(msg)
        else:
            # STARTTLS upgrade
            with smtplib.SMTP(smtp_host, smtp_port) as server:
                server.ehlo()
                server.starttls(context=context)
                server.ehlo()
                server.login(username, password)
                server.send_message(msg)
        print(f'Email sent to {msg["To"]}')
        return True
    except smtplib.SMTPAuthenticationError:
        print('ERROR: Authentication failed. Check your App Password.')
    except smtplib.SMTPRecipientsRefused as e:
        print(f'ERROR: Recipient refused: {e}')
    except smtplib.SMTPException as e:
        print(f'ERROR: SMTP error: {e}')
    except Exception as e:
        print(f'ERROR: Unexpected error: {e}')
    return False


# This will print the skip message if no password is set
send_email(plain_msg)

## 6. Complete Notification Function

In [ ]:
def notify(to: str, subject: str, body_text: str, body_html: str = None,
           attachments: list = None) -> bool:
    """
    High-level notification function.
    Builds the correct MIME structure and sends the email.

    Args:
        to           : Recipient email address
        subject      : Email subject line
        body_text    : Plain-text body (always required)
        body_html    : Optional HTML body
        attachments  : Optional list of file paths to attach

    Returns:
        True if sent successfully, False otherwise.
    """
    if attachments:
        outer = MIMEMultipart('mixed')
    elif body_html:
        outer = MIMEMultipart('alternative')
    else:
        return send_email(build_plain_email(to, subject, body_text))

    outer['From']    = formataddr((FROM_NAME, SMTP_USER))
    outer['To']      = to
    outer['Subject'] = subject
    outer['Date']    = formatdate(localtime=True)

    if body_html and attachments:
        alt = MIMEMultipart('alternative')
        alt.attach(MIMEText(body_text, 'plain', 'utf-8'))
        alt.attach(MIMEText(body_html, 'html',  'utf-8'))
        outer.attach(alt)
    elif body_html:
        outer.attach(MIMEText(body_text, 'plain', 'utf-8'))
        outer.attach(MIMEText(body_html, 'html',  'utf-8'))

    for fpath in (attachments or []):
        p = Path(fpath)
        with open(p, 'rb') as f:
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header('Content-Disposition', 'attachment', filename=p.name)
        outer.attach(part)

    return send_email(outer)


# Example usage
notify(
    to='team@example.com',
    subject='Scraping Job Complete',
    body_text='Job done. 1000 items scraped.',
    body_html='<b>Job done.</b> <span style="color:green">1000 items</span> scraped.',
    attachments=['/tmp/report.csv'],
)

## 7. Twilio SMS — Code Structure

Twilio lets you send SMS messages programmatically. You need a Twilio account (free trial available), an account SID, auth token, and a Twilio phone number.

In [ ]:
from twilio.rest import Client as TwilioClient

# Load Twilio credentials from environment
TWILIO_SID   = os.environ.get('TWILIO_ACCOUNT_SID', '')   # AC...
TWILIO_TOKEN = os.environ.get('TWILIO_AUTH_TOKEN',  '')   # your auth token
TWILIO_FROM  = os.environ.get('TWILIO_FROM_NUMBER', '')   # +1XXXXXXXXXX

def send_sms(to_number: str, message: str) -> str | None:
    """
    Send an SMS via Twilio.
    Returns the message SID on success, None on failure.
    Requires TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN, TWILIO_FROM_NUMBER env vars.
    """
    if not all([TWILIO_SID, TWILIO_TOKEN, TWILIO_FROM]):
        print('[SMS SKIPPED] Twilio credentials not set in environment.')
        print(f'  Would send to {to_number}: {message[:60]}')
        return None

    try:
        client = TwilioClient(TWILIO_SID, TWILIO_TOKEN)
        msg = client.messages.create(
            to=to_number,
            from_=TWILIO_FROM,
            body=message,
        )
        print(f'SMS sent: SID={msg.sid}  status={msg.status}')
        return msg.sid
    except Exception as e:
        print(f'SMS error: {e}')
        return None


# Demonstrate the call structure
send_sms('+15551234567', 'PyPath alert: scraping job completed successfully. 1000 items found.')

## 8. Email Sending Patterns

```python
# Pattern 1: Simple notification on job completion
def on_job_complete(job_name, stats):
    notify(
        to=os.environ['ALERT_EMAIL'],
        subject=f'[PyPath] Job complete: {job_name}',
        body_text=f'Job {job_name} done. Items: {stats["items"]}',
    )

# Pattern 2: Alert on error with traceback
import traceback
def on_error(job_name, exc):
    tb = traceback.format_exc()
    notify(
        to=os.environ['ALERT_EMAIL'],
        subject=f'[PyPath ERROR] {job_name}',
        body_text=f'Error in {job_name}:\n\n{tb}',
        body_html=f'<pre style="color:red">{tb}</pre>',
    )

# Pattern 3: Daily digest with attachment
def send_daily_digest(report_path):
    today = datetime.date.today().isoformat()
    notify(
        to=os.environ['ALERT_EMAIL'],
        subject=f'Daily Digest — {today}',
        body_text=f'See attached report for {today}.',
        attachments=[report_path],
    )
```

These patterns can be combined with the `schedule` library from Module 3.5.

## Practice Exercises

**Exercise 1 — Email Template Engine**  
Create a simple template function `render_email(template: str, **kwargs) -> str` that replaces `{{KEY}}` placeholders in a multi-line string. Then write two templates: one for a success notification and one for an error alert, and render both with sample data.

**Exercise 2 — Batch Notification**  
Write a function `send_bulk(recipients: list[dict], subject: str, template: str)` where each dict has `name` and `email` keys. The function should personalize the email body for each recipient (using their name), build a plain-text message, and call `send_email()` for each. Add a 0.5-second delay between sends to avoid rate limiting.

**Exercise 3 — Email Attachment from Memory**  
Modify the `add_attachment` function to also accept a `BytesIO` object (in-memory file) instead of a file path. This is useful when you generate a PDF or Excel file in memory (without saving to disk) and want to attach it directly. Test it by generating a small PDF with `reportlab` into a `BytesIO` and attaching it.